# Chapter 14 &mdash; $A_{TM}$, and Why RE Languages Are Called *Enumerable*

**Concept 10 of the Chapter 14 decomposition:** *$A_{TM}$, and Why RE Languages Are Called *Enumerable**

$A_{TM}$ is RE by simulation; and every RE language can be listed by a machine.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter14/Concept-A-TM-And-Enumerability/Concept-A-TM-And-Enumerability.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.LangDef        import *
from jove.Def_TM         import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


$$A_{TM} = \{\langle M,w\rangle : M \text{ accepts } w\}$$

**$A_{TM}$ is RE**: the semi-decider is the **universal machine** &mdash; simulate $M$ on
$w$ and accept if it does. On a non-member the simulation may run forever, which is
exactly the RE guarantee.

It is **not recursive** (Chapter 15, Concept 5), so by Concept 7 its complement is not
even RE.

**Why "enumerable"?** Because $L$ is RE iff some machine can **list** its members:
dovetail the recogniser over all strings, printing each one as it is accepted. The
order is not lexicographic &mdash; you print what finishes first &mdash; but every member
eventually appears. A language is RE exactly when it can be enumerated.

## 2. Definitions

### A universal simulator, in miniature

In [ ]:
# We simulate "M accepts w" with Python callables standing in for machines.
MACHINES = {
 'M_starts1' : lambda w: w.startswith('1'),
 'M_even0'   : lambda w: w.count('0') % 2 == 0,
 'M_loops'   : None,                # stands for a machine that never halts
}

def universal(mname, w, fuel=50):
    f = MACHINES[mname]
    if f is None:
        return None                 # simulation runs forever: no answer
    return f(w)

### An enumerator: dovetail the recogniser over all strings

In [ ]:
from itertools import product
def enumerate_re(recognize, cap=60, sigma='01'):
    # dovetailing: (string index, fuel) pairs in diagonal order
    strs = [''.join(p) for k in range(6) for p in product(sigma, repeat=k)]
    out = []
    for total in range(cap):
        for i in range(total + 1):
            fuel = total - i
            if i < len(strs) and recognize(strs[i], fuel) and strs[i] not in out:
                out.append(strs[i])
    return out

## 3. Tests

The universal machine semi-decides $A_{TM}$.

In [ ]:
for m, w in [('M_starts1', '101'), ('M_starts1', '011'), ('M_even0', '0011')]:
    print("  <%s, %r> : %s" % (m, w, universal(m, w)))
assert universal('M_starts1', '101') is True
assert universal('M_starts1', '011') is False

On some pairs the simulation never returns &mdash; the RE guarantee, exactly.

In [ ]:
print("  <M_loops, '1'> :", universal('M_loops', '1'), " <- no answer, ever")
assert universal('M_loops', '1') is None
print()
print("Accepting pairs are found.  Non-accepting pairs may simply never report.")

**Enumeration.** Dovetailing lists every member, in a strange order.

In [ ]:
def rec_even0(w, fuel):
    return len(w) <= fuel and w.count('0') % 2 == 0
listed = enumerate_re(rec_even0)
print("enumerated :", listed[:12])
assert all(w.count('0') % 2 == 0 for w in listed)

Every member eventually appears &mdash; that is what 'enumerable' means.

In [ ]:
want = {''.join(p) for k in range(4) for p in product('01', repeat=k)
        if ''.join(p).count('0') % 2 == 0}
got = set(enumerate_re(rec_even0, cap=80))
print("expected %d short members, listed %d of them" % (len(want), len(want & got)))
assert want <= got
print("\nThe ORDER is not lexicographic -- it is 'whatever finishes first'.")

Why enumeration must dovetail rather than go string by string.

In [ ]:
print("string-by-string : run the recogniser on '' to completion, then '0', ...")
print("                   -- hangs forever at the first non-member")
print()
print("dovetailed       : give string i a little more fuel each round")
print("                   -- every member is eventually reached")

And the definition falls out.

In [ ]:
print("L is RE  <=>  some TM recognises L")
print("         <=>  some TM enumerates L")
print()
print("The two are interchangeable, which is why the family has two names.")

## 4. Exercises


1. Write the enumerator for $\Sigma^*$. Is it lexicographic?
2. If $L$ can be enumerated **in lexicographic order**, is $L$ recursive?
3. Why does the universal machine exist at all? What does it need to parse?

In [ ]:
# Your work for the exercises above.